# Clinical Trial Safety Analysis Pipeline

## Real-World Cell Differentiation in Drug Development

In drug development, clinical trial data passes through multiple specialist reviewers
before a regulatory submission. Each reviewer applies domain-specific expertise:

```
                    STEM Agent (Generic)
                           │
          ┌────────────────┼────────────────┐────────────────┐
          │                │                │                │
          ▼                ▼                ▼                ▼
    Literature        Biostatistician  Safety           Regulatory
    Reviewer          (power analysis) Reviewer         Writer
    (reflexion)       (CoT, precise)   (react, strict)  (CoT, formal)
```

This notebook differentiates a single STEM Agent into **four medical domain
specialists** and routes a clinical trial dataset through each one.

### Differentiation Axes

| Persona | Strategy | Key Trait | Forbidden Topics |
|---------|----------|-----------|------------------|
| Literature Reviewer | reflexion | High tool use (arXiv/PubMed) | Specific dosing recommendations |
| Biostatistician | chain_of_thought | High confidence (0.9) | Causal claims without RCT evidence |
| Safety Reviewer | react | Low creativity, high precision | Downplaying adverse events |
| Regulatory Writer | chain_of_thought | High verbosity, formal | Promotional language |

## Setup

In [ ]:
import sys, os, time, json

sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, '.')

from stem_client import (
    StemAgentClient, print_persona, compare_responses,
    bar_chart, radar_text, timed
)

client = StemAgentClient()
server_up = client.ensure_running()

## The Clinical Trial Dataset

A Phase III randomized controlled trial for a novel GLP-1 receptor agonist
("QuantaGlide") for Type 2 diabetes management. The trial data needs
multi-perspective review before FDA submission.

In [ ]:
TRIAL_DATA = {
    "trial_id": "NCT-2024-QG301",
    "drug": "QuantaGlide (QG-401) — GLP-1 receptor agonist",
    "indication": "Type 2 Diabetes Mellitus (T2DM) in adults with inadequate glycemic control on metformin",
    "phase": "Phase III, multi-center, double-blind, placebo-controlled RCT",
    "enrollment": "N=2,847 (randomized 1:1:1 — QG-401 high dose : QG-401 low dose : placebo)",
    "duration": "52 weeks treatment + 12 weeks follow-up",
    "primary_endpoint": "Change in HbA1c from baseline at Week 52",
    "results": {
        "high_dose": {"hba1c_change": -1.8, "ci_95": "(-2.0, -1.6)", "p_value": "<0.001"},
        "low_dose": {"hba1c_change": -1.2, "ci_95": "(-1.4, -1.0)", "p_value": "<0.001"},
        "placebo": {"hba1c_change": -0.3, "ci_95": "(-0.5, -0.1)", "p_value": "ref"},
    },
    "secondary_endpoints": {
        "weight_change_kg": {"high": -5.2, "low": -3.1, "placebo": -0.8},
        "fasting_glucose_mgdl": {"high": -42, "low": -28, "placebo": -8},
        "systolic_bp_mmhg": {"high": -4.1, "low": -2.3, "placebo": -0.5},
    },
    "safety": {
        "serious_adverse_events": {"high": "4.2%", "low": "3.1%", "placebo": "2.8%"},
        "nausea": {"high": "31%", "low": "22%", "placebo": "8%"},
        "vomiting": {"high": "14%", "low": "9%", "placebo": "3%"},
        "diarrhea": {"high": "18%", "low": "12%", "placebo": "6%"},
        "pancreatitis": {"high": "0.4% (4 cases)", "low": "0.1% (1 case)", "placebo": "0.1% (1 case)"},
        "thyroid_c_cell": "No signal (monitored per FDA requirement for GLP-1 class)",
        "discontinuation_due_to_ae": {"high": "8.3%", "low": "5.1%", "placebo": "2.2%"},
    },
}

def format_trial_brief():
    lines = ["CLINICAL TRIAL DATA:"]
    for k, v in TRIAL_DATA.items():
        if isinstance(v, dict):
            lines.append(f"  {k}:")
            for k2, v2 in v.items():
                lines.append(f"    {k2}: {v2}")
        else:
            lines.append(f"  {k}: {v}")
    return "\n".join(lines)

print(format_trial_brief())

## Define the Four Medical Domain Personas

Each persona is a **differentiation signal** — it transforms the generic STEM Agent
into a domain specialist with specific:
- Reasoning strategy (how it thinks)
- Behavior parameters (how deep, creative, verbose)
- Safety constraints (forbidden topics)
- Tool access (which MCP servers it can use)

In [ ]:
PERSONAS = {
    "literature_reviewer": {
        "name": "LiteratureReviewer",
        "systemPrompt": (
            "You are a medical literature reviewer specializing in endocrinology "
            "and diabetes pharmacotherapy. Search for and synthesize relevant "
            "clinical evidence. Compare trial results against existing GLP-1 RA "
            "data (semaglutide, liraglutide, tirzepatide). Cite specific trials "
            "by NCT ID or publication. Flag any discrepancies with established evidence."
        ),
        "allowedIntents": ["question", "analysis_request"],
        "forbiddenTopics": [
            "specific dosing recommendations for individual patients",
            "off-label use suggestions",
        ],
        "preferredStrategy": "reflexion",
        "defaultBehavior": {
            "reasoningDepth": 5,
            "confidenceThreshold": 0.7,
            "creativityLevel": 0.2,
            "toolUsePreference": 0.9,
            "verbosityLevel": 0.7,
        },
        "requiredMCPServers": ["arxiv", "pubmed-mcp"],
        "toolAllowlist": ["search_arxiv", "search_pubmed", "get_trial_data"],
        "domainTags": ["medical", "literature", "endocrinology", "clinical-trials"],
    },
    "biostatistician": {
        "name": "Biostatistician",
        "systemPrompt": (
            "You are a senior biostatistician reviewing clinical trial methodology "
            "and results. Evaluate: sample size adequacy, randomization quality, "
            "statistical test appropriateness, multiplicity adjustment, effect size "
            "clinical significance, and handling of missing data. Use precise "
            "statistical language and quantify uncertainty."
        ),
        "allowedIntents": ["analysis_request", "question"],
        "forbiddenTopics": [
            "causal claims without randomized controlled trial evidence",
            "p-hacking or selective outcome reporting",
        ],
        "preferredStrategy": "chain_of_thought",
        "defaultBehavior": {
            "reasoningDepth": 5,
            "confidenceThreshold": 0.9,
            "creativityLevel": 0.1,
            "toolUsePreference": 0.6,
            "verbosityLevel": 0.6,
        },
        "requiredMCPServers": [],
        "toolAllowlist": [],
        "domainTags": ["statistics", "biostatistics", "clinical-trials", "methodology"],
    },
    "safety_reviewer": {
        "name": "SafetyReviewer",
        "systemPrompt": (
            "You are an FDA-experienced drug safety reviewer. Evaluate adverse event "
            "profiles, serious adverse event rates, dose-response relationships for "
            "toxicity, class-effect safety signals, and benefit-risk balance. Flag "
            "any safety signals that require REMS, black box warnings, or additional "
            "post-marketing surveillance. Be conservative — patient safety is paramount."
        ),
        "allowedIntents": ["analysis_request", "debugging"],
        "forbiddenTopics": [
            "downplaying or minimizing adverse events",
            "comparing safety favorably without statistical evidence",
        ],
        "preferredStrategy": "react",
        "defaultBehavior": {
            "reasoningDepth": 4,
            "confidenceThreshold": 0.85,
            "creativityLevel": 0.05,
            "toolUsePreference": 0.7,
            "verbosityLevel": 0.5,
        },
        "requiredMCPServers": ["faers-mcp"],
        "toolAllowlist": ["query_faers", "check_drug_interactions", "get_safety_alerts"],
        "domainTags": ["safety", "pharmacovigilance", "adverse-events", "fda"],
    },
    "regulatory_writer": {
        "name": "RegulatoryWriter",
        "systemPrompt": (
            "You are a regulatory medical writer preparing documents for FDA "
            "submission (NDA/BLA). Write in formal regulatory language following "
            "ICH E3 guidelines for clinical study reports. Structure content per "
            "CTD Module 2.5 (Clinical Overview) format. Be comprehensive, precise, "
            "and never use promotional language."
        ),
        "allowedIntents": ["creative", "analysis_request"],
        "forbiddenTopics": [
            "promotional or marketing language",
            "superlatives without evidence",
            "comparative superiority claims without head-to-head trials",
        ],
        "preferredStrategy": "chain_of_thought",
        "defaultBehavior": {
            "reasoningDepth": 4,
            "confidenceThreshold": 0.8,
            "creativityLevel": 0.3,
            "toolUsePreference": 0.4,
            "verbosityLevel": 0.9,
        },
        "requiredMCPServers": [],
        "toolAllowlist": [],
        "domainTags": ["regulatory", "medical-writing", "fda", "ich", "nda"],
    },
}

for role, persona in PERSONAS.items():
    print(f"[{role.upper()}]")
    print_persona(persona)

## Behavioral Fingerprints

Each persona produces a unique behavioral profile. Notice how:
- **SafetyReviewer** has the lowest creativity (0.05) — no room for speculation
- **Biostatistician** has the highest confidence threshold (0.9) — only reports high-confidence findings
- **LiteratureReviewer** has the highest tool use (0.9) — needs to search external databases
- **RegulatoryWriter** has the highest verbosity (0.9) — FDA documents require comprehensive detail

In [ ]:
print("BEHAVIORAL FINGERPRINTS\n")
for persona in PERSONAS.values():
    radar_text(persona)
    print()

print("\nBEHAVIOR COMPARISON\n")
chart_data = {p["name"]: p["defaultBehavior"] for p in PERSONAS.values()}
bar_chart(chart_data)

## Safety Constraints (Forbidden Topics)

A critical feature of cell differentiation: each specialist has **guardrails**
that prevent it from operating outside its safe scope.

| Specialist | Cannot Do |
|---|---|
| Literature Reviewer | Recommend specific doses for patients |
| Biostatistician | Make causal claims without RCT evidence |
| Safety Reviewer | Downplay or minimize adverse events |
| Regulatory Writer | Use promotional or marketing language |

In [ ]:
print("SAFETY CONSTRAINTS BY PERSONA\n")
for role, persona in PERSONAS.items():
    forbidden = persona.get("forbiddenTopics", [])
    tools = persona.get("toolAllowlist", [])
    mcp = persona.get("requiredMCPServers", [])
    
    print(f"  {persona['name']}:")
    print(f"    Forbidden:  {'; '.join(forbidden) if forbidden else '(none)'}")
    print(f"    Tools:      {', '.join(tools) if tools else '(unrestricted)'}")
    print(f"    MCP:        {', '.join(mcp) if mcp else '(none required)'}")
    print()

## Live Differentiation: Same Trial Data, Four Specialists

Each specialist receives the **same clinical trial dataset** but produces
fundamentally different analysis based on their persona configuration.

In [ ]:
trial_brief = format_trial_brief()

PROMPTS = {
    "literature_reviewer": (
        f"Review this clinical trial data and compare against existing GLP-1 RA evidence.\n\n"
        f"{trial_brief}\n\n"
        "How do these results compare to SUSTAIN, PIONEER, and SURPASS trials? "
        "Identify any results that deviate significantly from class expectations."
    ),
    "biostatistician": (
        f"Evaluate the statistical rigor of this clinical trial.\n\n"
        f"{trial_brief}\n\n"
        "Assess: sample size adequacy, effect size clinical significance, "
        "confidence interval interpretation, multiplicity concerns for secondary "
        "endpoints, and missing data handling."
    ),
    "safety_reviewer": (
        f"Conduct a safety review of this clinical trial data.\n\n"
        f"{trial_brief}\n\n"
        "Evaluate: adverse event profile relative to class, dose-response toxicity, "
        "pancreatitis signal, discontinuation rates, and whether a REMS or boxed "
        "warning may be warranted. Provide benefit-risk assessment."
    ),
    "regulatory_writer": (
        f"Draft a Clinical Overview section (CTD Module 2.5 style) for this trial.\n\n"
        f"{trial_brief}\n\n"
        "Structure as: Background, Study Design, Efficacy Results, Safety Profile, "
        "Benefit-Risk Conclusion. Use formal regulatory language per ICH E3."
    ),
}

# These are complex medical analysis prompts — give the LLM plenty of time
CALL_TIMEOUT = 300  # 5 minutes per specialist

responses = {}
timings = {}

if server_up:
    for role, prompt in PROMPTS.items():
        persona = PERSONAS[role]
        caller_id = f"trial-{role}"
        print(f"Dispatching to {persona['name']}...", end=" ", flush=True)

        try:
            start = time.perf_counter()
            resp = client.chat(prompt, caller_id=caller_id, timeout=CALL_TIMEOUT)
            elapsed = time.perf_counter() - start

            responses[role] = resp
            timings[role] = elapsed

            words = len(str(resp.get('content', '')).split())
            print(f"[{resp.get('status', '?')}] {words} words in {elapsed:.1f}s")
        except Exception as e:
            elapsed = time.perf_counter() - start
            timings[role] = elapsed
            print(f"[TIMEOUT after {elapsed:.0f}s] {type(e).__name__}: {e}")
            print(f"  Tip: increase CALL_TIMEOUT or check server load")

    succeeded = len(responses)
    total = len(PROMPTS)
    print(f"\n{succeeded}/{total} specialists completed successfully.")
else:
    print("Server not running — see expected output description below.")
    print("  Literature Reviewer: Comparative analysis against SUSTAIN/PIONEER/SURPASS")
    print("  Biostatistician: Power analysis, CI interpretation, multiplicity concerns")
    print("  Safety Reviewer: AE profile analysis, pancreatitis signal, REMS assessment")
    print("  Regulatory Writer: Formal CTD Module 2.5 clinical overview draft")

## Response Comparison

Notice how each specialist focuses on entirely different aspects of the same data:

In [ ]:
if responses:
    resp_list = list(responses.values())
    labels = [PERSONAS[r]["name"] for r in responses.keys()]
    compare_responses(resp_list, labels)
    
    print("\nRESPONSE METRICS:")
    print(f"{'Specialist':<22s} {'Words':>8s} {'Time':>8s} {'Trace':>8s}")
    print("-" * 50)
    for role, resp in responses.items():
        name = PERSONAS[role]["name"]
        words = len(str(resp.get('content', '')).split())
        trace = len(resp.get('reasoningTrace') or resp.get('reasoning_trace') or [])
        t = timings[role]
        print(f"{name:<22s} {words:>8d} {t:>7.1f}s {trace:>8d}")
else:
    print("No live responses available. Start the server and re-run.")

## Caller Profile Adaptation

After each interaction, the agent's **Exponential Moving Average (EMA)** system
updates each caller's profile across 20+ dimensions. Over many interactions,
the profiles diverge — the Literature Reviewer caller develops a different
behavioral fingerprint than the Safety Reviewer caller.

This is analogous to **epigenetic memory** in biology: cells accumulate
modifications that reinforce their specialization over time.

In [ ]:
if server_up:
    print("CALLER PROFILES (Epigenetic Memory)\n")
    for role in PERSONAS:
        caller_id = f"trial-{role}"
        try:
            profile = client.get_profile(caller_id)
            style = profile.get("style", {})
            interactions = profile.get("interactionCount", 0)
            
            print(f"  {PERSONAS[role]['name']} ({caller_id}):")
            print(f"    Interactions:    {interactions}")
            for k, v in sorted(style.items()):
                print(f"    {k:18s}: {v}")
        except Exception as e:
            print(f"  {PERSONAS[role]['name']}: (profile not established yet)")
        print()
    
    print("With more interactions, these profiles diverge further,")
    print("reinforcing each specialist's behavioral patterns.")
else:
    print("Start the server and run the live differentiation cells above")
    print("to see caller profiles evolve.")

## Sequential Pipeline: Literature → Statistics → Safety → Regulatory

In a real regulatory submission, findings flow between specialists.
The Literature Reviewer's output informs the Biostatistician, whose
analysis feeds the Safety Reviewer, and finally the Regulatory Writer
synthesizes everything into a formal document.

In [ ]:
if server_up and responses:
    # Use the literature review as context for a deeper statistical analysis
    lit_output = str(responses.get("literature_reviewer", {}).get("content", ""))
    
    pipeline_prompt = (
        "Based on the literature reviewer's comparative analysis below, "
        "write a 200-word regulatory benefit-risk conclusion that integrates "
        "efficacy context, statistical confidence, and safety signals.\n\n"
        f"Literature Review:\n{lit_output[:2000]}\n\n"
        "Write the conclusion in ICH E1/E3 format."
    )
    
    print("[INTEGRATED PIPELINE] Literature → Regulatory Writer\n")
    try:
        start = time.perf_counter()
        pipeline_resp = client.chat(
            pipeline_prompt, caller_id="trial-regulatory_writer", timeout=CALL_TIMEOUT
        )
        elapsed = time.perf_counter() - start
        
        content = str(pipeline_resp.get("content", ""))
        print(f"Status: {pipeline_resp.get('status', '?')} ({elapsed:.1f}s)\n")
        print(content[:1500])
    except Exception as e:
        print(f"Pipeline call failed: {type(e).__name__}: {e}")
        print("Try increasing CALL_TIMEOUT in the live-differentiation cell.")
else:
    print("Run the live differentiation cells above first,")
    print("then this cell will chain the outputs into a pipeline.")

## Summary: Biology ↔ STEM Agent Mapping

| Biological Concept | STEM Agent Equivalent | This Example |
|---|---|---|
| Stem cell | Generic agent (no persona) | Base STEM Agent server |
| Differentiation signal | DomainPersona JSON config | 4 medical persona definitions |
| Gene expression levels | Behavior parameters | Creativity=0.05 for Safety, 0.9 verbosity for Regulatory |
| Cell type | Specialized agent | Literature Reviewer, Biostatistician, Safety Reviewer, Regulatory Writer |
| Cell surface markers | Domain tags | `pharmacovigilance`, `biostatistics`, `regulatory` |
| Metabolic pathway | Reasoning strategy | Reflexion for literature search, React for safety triage |
| Immune tolerance | Forbidden topics | Safety Reviewer can't downplay AEs |
| Cell membrane channels | Tool allowlist | Only Literature Reviewer can access PubMed |
| Epigenetic memory | Caller profile EMA | Profiles diverge with each interaction |

### Production Deployment

For a real pharmaceutical company:
```bash
# Dedicated safety review server (locked to safety persona)
DOMAIN_PERSONA=domains/pharma-safety/persona.json \
  node --env-file=.env dist/server.js

# Dedicated regulatory writing server
DOMAIN_PERSONA=domains/regulatory-writing/persona.json \
  node --env-file=.env dist/server.js
```

Each instance is fully differentiated with locked-down safety constraints
and tool access — exactly like how specialized cells in an organism can
only perform their designated functions.